# 06 — Advise on each rule's detection threshold

The threshold job. One model call per registered critical data element, advising on
the `fail_threshold_pct` of every active rule that monitors it, and writing one row per
rule to `results.threshold_proposal`. A reviewer decides each proposal in the app's
Thresholds page; adopting one appends a new `rule_version`. **Nothing here changes a
rule.**

**This is detection advice, not triage.** It reads the CDE register, the rule registry
and the run history. It does not read a cohort, a hypothesis or the disposition
register, and notebook 03 does not read this. A limit is a property of a rule; a
problem is a property of a run.

**Two bases, and the brief carries both, labelled.** The tolerance the register
declares on the element — `config.cde_registry.tolerance_pct`, what the business will
accept — and where `violation_pct` has sat across every run: min, median, p90, max, and
how many runs breached the current limit. The first is a **ceiling**: `validate` refuses
a proposal above it, and so does a CHECK constraint on the table. Without that,
"raise the limit until the check passes" is a valid answer, and it is the answer a model
asked about limits will reach for. Where no tolerance is declared the expected advice
is `unchanged`: history alone says where the data is, not where it may be.

**`current_threshold_pct` is copied from the registry, never from the response.** The
"from" half of a proposal is a fact whatever the model echoed.

### Before running

Runs as the **threshold job's** service principal. It needs `SELECT` on `config` and
`results` and `MODIFY` on `results.threshold_proposal` alone — the same shape as the
triage job's grant on `results.cohort`, and like it, not in `07_grants.sql`: giving a
job principal a write is a decision, not a schema change.


In [ ]:
from openai import OpenAI
import json, re, uuid, hashlib
import pandas as pd
from datetime import datetime

# TWO LAYOUTS, same as notebook 03. Change SANDPIT, not the constants.
SANDPIT = True

if SANDPIT:
    CATALOG, SCHEMA, PREFIX = "workspace", "dq_triage", "dq_"
    def _t(group, name): return f"{CATALOG}.{SCHEMA}.{PREFIX}{group}_{name}"
else:
    CATALOG = "dq"
    def _t(group, name): return f"{CATALOG}.{group}.{name}"

CHECK_RUN      = _t("results", "check_run")
PROPOSAL       = _t("results", "threshold_proposal")
PROPOSAL_CUR   = _t("results", "v_threshold_proposal_current")
RULE_CURRENT   = _t("config",  "v_rule_registry_current")
CDE_CURRENT    = _t("config",  "v_cde_registry_current")

MODEL_NAME  = "system.ai.gpt-oss-120b"
MAX_TOKENS  = 1800
TEMPERATURE = 0.0     # advice that lands in a table a reviewer decides on should be reproducible

NS = uuid.UUID("6f1b4c2e-0000-4000-8000-000000000002")
def det_uuid(*parts): return str(uuid.uuid5(NS, "|".join(parts)))


In [ ]:
def _ctx():
    return dbutils.notebook.entry_point.getDbutils().notebook().getContext()


API_ROOT  = _ctx().apiUrl().get()
API_TOKEN = _ctx().apiToken().get()
client = OpenAI(api_key=API_TOKEN, base_url=f"{API_ROOT}/ai-gateway/mlflow/v1")

# The Lakeflow run id, the portable way: submit with
#     "base_parameters": {"job_run_id": "{{job.run_id}}"}
# currentRunId() is not whitelisted on serverless -- see notebook 03.
dbutils.widgets.text("job_run_id", "")
_param = dbutils.widgets.get("job_run_id").strip()
if _param and not _param.startswith("{{"):
    JOB_RUN_ID = _param
else:
    try:
        JOB_RUN_ID = _ctx().currentRunId().toString()
    except Exception as exc:
        JOB_RUN_ID = f"interactive-{uuid.uuid4()}"
        print(f"no job run id ({type(exc).__name__}) - stamping {JOB_RUN_ID}")

PROPOSED_TS = datetime.now().replace(microsecond=0)
PROPOSAL_RUN_ID = det_uuid("threshold-run", JOB_RUN_ID, PROPOSED_TS.isoformat())
print("Unity Gateway client created ·", MODEL_NAME)
print("job_run_id ·", JOB_RUN_ID, "· proposal_run_id ·", PROPOSAL_RUN_ID)


In [ ]:
# Smoke test -- confirms the endpoint answers before any worklist is built.
print(client.responses.create(
    model=MODEL_NAME, input="Reply with the single word: ready", max_output_tokens=20
).output_text)


---
## 1 · The worklist is the CDE register

One brief per registered element, covering every active or shadow rule that names it.
A rule that names no registered element is not on the worklist — `cde_id` is NOT NULL
on the registry, so there is no such rule, and an element retired from the register
takes its rules off the list with it.


In [ ]:
worklist = spark.sql(f"""
    SELECT e.cde_id, e.cde_name, e.data_class, e.criticality, e.pii, e.tolerance_pct,
           e.definition, e.regulatory_basis, e.owner_group AS element_owner,
           r.rule_id, r.rule_version, r.rule_name, r.rule_type, r.target_table,
           r.target_column, r.rule_expr, r.scope_filter, r.fail_threshold_pct,
           r.severity, r.status, r.owner_group
    FROM   {CDE_CURRENT} e
    JOIN   {RULE_CURRENT} r ON r.cde_id = e.cde_id
    WHERE  e.status = 'registered' AND r.status IN ('active', 'shadow')
    ORDER  BY e.cde_id, r.rule_id
""").toPandas()

# Where each rule's violation rate has sat, over every run it produced a verdict on.
# pass and breach only: a skipped or errored run has no rate.
history = spark.sql(f"""
    SELECT rule_id,
           COUNT(*)                                             AS runs_observed,
           MIN(violation_pct)                                   AS pct_min,
           PERCENTILE(violation_pct, 0.5)                       AS pct_median,
           PERCENTILE(violation_pct, 0.9)                       AS pct_p90,
           MAX(violation_pct)                                   AS pct_max,
           SUM(CASE WHEN status = 'breach' THEN 1 ELSE 0 END)   AS runs_breaching
    FROM   {CHECK_RUN}
    WHERE  status IN ('pass', 'breach')
    GROUP  BY rule_id
""").toPandas().set_index("rule_id")

latest = spark.sql(f"""
    SELECT rule_id, violation_pct, violation_count, rows_scanned, status
    FROM   {CHECK_RUN}
    WHERE  run_ts = (SELECT MAX(run_ts) FROM {CHECK_RUN})
""").toPandas().set_index("rule_id")

print(f"{worklist.cde_id.nunique()} elements · {len(worklist)} rules · "
      f"{worklist.tolerance_pct.notna().sum()} with a declared tolerance")
display(worklist[["cde_id", "criticality", "tolerance_pct", "rule_id", "fail_threshold_pct", "status"]])


---
## 2 · The brief

One page per element. Per rule: the current limit, the element's declared tolerance
(the ceiling), and where the rate has sat. The rule's expression and scope are there so
the model can tell a rule whose breach is scope from one whose breach is level — the
first is never answered with a limit.


In [ ]:
def _blank(v, alt=None):
    return alt if v is None or (isinstance(v, float) and v != v) else v


def _stat(rid):
    if rid not in history.index:
        return None
    h = history.loc[rid]
    return {"runs_observed": int(h.runs_observed), "pct_min": round(float(h.pct_min), 4),
            "pct_median": round(float(h.pct_median), 4), "pct_p90": round(float(h.pct_p90), 4),
            "pct_max": round(float(h.pct_max), 4), "runs_breaching": int(h.runs_breaching)}


def build_payload(cde_id):
    g = worklist[worklist.cde_id == cde_id]
    e = g.iloc[0]
    members = []
    for _, r in g.iterrows():
        lt = latest.loc[r.rule_id] if r.rule_id in latest.index else None
        members.append({
            "rule_id": r.rule_id, "rule_version": int(r.rule_version),
            "rule_name": r.rule_name, "rule_type": r.rule_type,
            "target_table": r.target_table, "target_column": _blank(r.target_column),
            "rule_expr": r.rule_expr, "scope_filter": _blank(r.scope_filter),
            "severity": r.severity, "status": r.status, "owner_group": r.owner_group,
            "current_threshold_pct": float(r.fail_threshold_pct),
            "history": _stat(r.rule_id),
            "latest_violation_pct": (None if lt is None else round(float(lt.violation_pct), 4)),
            "latest_status": (None if lt is None else lt.status),
        })
    tol = _blank(e.tolerance_pct)
    return {
        "cde_id": cde_id, "cde_name": e.cde_name, "data_class": e.data_class,
        "criticality": e.criticality, "pii": bool(e.pii),
        "tolerance_pct": None if tol is None else float(tol),
        "definition": e.definition, "regulatory_basis": _blank(e.regulatory_basis),
        "element_owner": e.element_owner,
        "members": members,
    }


def rule_block(m):
    target = (f"{m['target_table']}.{m['target_column']}" if m["target_column"]
              else f"{m['target_table']} (cross-table rule, no single target column)")
    h = m["history"]
    hist = (f"over {h['runs_observed']} runs the violation rate was min {h['pct_min']}% / "
            f"median {h['pct_median']}% / p90 {h['pct_p90']}% / max {h['pct_max']}%, "
            f"breaching the current limit on {h['runs_breaching']}" if h else "no run history")
    return f"""
  Rule {m['rule_id']} (version {m['rule_version']}) - {m['rule_name']}
    type       : {m['rule_type']}   severity: {m['severity']}   status: {m['status']}   owner: {m['owner_group']}
    target     : {target}
    expression : {m['rule_expr']}
    scope      : {m['scope_filter'] or '(none - the rule is applied to every row of the table)'}
    limit now  : {m['current_threshold_pct']}%
    latest run : {m['latest_violation_pct']}% ({m['latest_status']})
    history    : {hist}"""


def build_brief(p):
    tol = (f"{p['tolerance_pct']}% of rows in scope may violate - THE CEILING for every rule below"
           if p["tolerance_pct"] is not None else
           "(none declared - history alone is not a basis for raising any limit below)")
    return f"""
Advise on the detection threshold of every rule monitoring one critical data element.

ELEMENT
-------
{p['cde_id']} - {p['cde_name']}  ({p['data_class']}, criticality {p['criticality']}, PII {p['pii']})
  definition         : {p['definition']}
  regulatory basis   : {p['regulatory_basis'] or '(none recorded)'}
  owning group       : {p['element_owner']}
  declared tolerance : {tol}

RULES MONITORING IT
-------------------{''.join(rule_block(m) for m in p['members'])}

TASK
----
For EVERY rule above, advise on its fail_threshold_pct: the limit you would set, on what
basis, why, and which team should review it. The declared tolerance is a ceiling. "Keep
it" is a valid answer and usually the right one; say why in figures.

Return only the requested JSON object.
"""


payloads = [build_payload(c) for c in worklist.cde_id.unique()]
print(f"{len(payloads)} briefs assembled")
print(build_brief(payloads[0]))


In [ ]:
SYSTEM_PROMPT = '''
You are an experienced enterprise Data Quality Analyst advising on the detection
thresholds of data quality rules. A threshold (fail_threshold_pct) is the violation
rate at or above which a rule's run is a breach. It separates normal variation from a
breach; it is never a remedy.

1.  Base your advice only on the evidence supplied. Do not invent facts.
2.  Two bases exist and the brief labels both. The DECLARED TOLERANCE on the element is
    what the business will accept: a CEILING. Never propose a limit above it. The RUN
    HISTORY is where the violation rate has sat: where the data is, not where it may be.
3.  Set basis to element_tolerance, run_history or both according to what the proposal
    rests on, or to unchanged with proposed_threshold_pct exactly equal to the current
    limit. Most rules should be unchanged; say why in figures.
4.  Never set a limit at the level the data currently sits at so that the check passes.
    That hides a defect; it does not set a limit.
5.  A rule whose breach is SCOPE - it fires on rows that legitimately lack the value, and
    a scope_filter would return zero - is never answered with a limit. Say unchanged and
    say the remedy is the scope.
6.  Where no tolerance is declared, history alone is not a basis for raising anything.
    Say unchanged and say that a tolerance has to be declared first.
7.  A limit may be LOWERED on run_history alone: a rule whose history never approaches
    its limit can be tightened, and that is the one case history decides by itself.
8.  Where the declared tolerance is above where the data sits and the rule is breaching
    at zero, the tolerance is the honest limit - the business has said this share may be
    wrong - and the reviewer's question is whether the tolerance is really agreed.
9.  rationale quotes the figures it rests on. reviewer is the team that should decide -
    a team, not a person; the rule's owning group unless there is a reason otherwise.
10. You advise. Nothing you return changes a rule; a person adopts a proposal and a new
    rule version is appended. Do not propose that the system change anything itself.
11. Return only valid JSON, exactly this structure, with one entry per rule in the brief:

{
  "proposals": [
    {"rule_id": "rule_id", "proposed_threshold_pct": 0.0,
     "basis": "element_tolerance | run_history | both | unchanged",
     "rationale": "the figures this rests on", "reviewer": "team to review it"}
  ],
  "reasoning": "brief explanation linking evidence to the proposals"
}
'''


---
## 3 · Call, validate, and drop what does not validate

`validate` is the gate. A response above the element's tolerance, or missing a rule, or
calling a change `unchanged`, is **dropped for the whole element, not written** — a bad
model answer is a defect, not advice. The same ceiling is a CHECK constraint on the
table, stated twice because a prompt can drift without anyone noticing.


In [ ]:
BASES = ["element_tolerance", "run_history", "both", "unchanged"]


def parse_llm_json(text):
    text = re.sub(r"\s*```$", "", re.sub(r"^```(json)?\s*", "", text.strip()))
    return json.loads(text)


def validate(rec, payload):
    problems = []
    items = rec.get("proposals")
    if not isinstance(items, list):
        return ["proposals missing or not a list"]
    members = {m["rule_id"]: m for m in payload["members"]}
    named = [str(t.get("rule_id")) for t in items if isinstance(t, dict)]
    if sorted(named) != sorted(members):
        problems.append(f"proposals cover {named}, rules are {sorted(members)}")
    ceiling = payload["tolerance_pct"]
    for t in items:
        if not isinstance(t, dict):
            problems.append(f"not an object: {t!r}"); continue
        rid = str(t.get("rule_id")); m = members.get(rid)
        if m is None:
            continue
        try:
            pct = float(t.get("proposed_threshold_pct"))
        except (TypeError, ValueError):
            problems.append(f"{rid}: proposed_threshold_pct not numeric"); continue
        if not 0.0 <= pct <= 100.0:
            problems.append(f"{rid}: {pct} out of range")
        basis = t.get("basis")
        if basis not in BASES:
            problems.append(f"{rid}: basis {basis!r} not in enum")
        elif (basis == "unchanged") != (pct == m["current_threshold_pct"]):
            problems.append(f"{rid}: basis {basis!r} but {pct} vs current {m['current_threshold_pct']}")
        if not str(t.get("rationale") or "").strip():
            problems.append(f"{rid}: no rationale")
        if not str(t.get("reviewer") or "").strip():
            problems.append(f"{rid}: no reviewer")
        # THE CEILING. The one check that matters.
        if ceiling is not None and pct > float(ceiling):
            problems.append(f"{rid}: {pct}% is above the {ceiling}% tolerance the element declares")
        if ceiling is None and basis != "unchanged" and pct > m["current_threshold_pct"]:
            problems.append(f"{rid}: raised on history alone with no declared tolerance (rule 6)")
    if not str(rec.get("reasoning") or "").strip():
        problems.append("missing reasoning")
    return problems


def advise(payload):
    brief = build_brief(payload)
    raw = client.responses.create(
        model=MODEL_NAME, instructions=SYSTEM_PROMPT, input=brief,
        temperature=TEMPERATURE, max_output_tokens=MAX_TOKENS,
    ).output_text
    rec = parse_llm_json(raw)
    problems = validate(rec, payload)
    if problems:
        raise ValueError("; ".join(problems))
    return rec, brief


advised, rejected = [], []
for i, p in enumerate(payloads, 1):
    print(f"[{i}/{len(payloads)}] {p['cde_id']} ({len(p['members'])} rule(s))", end=" ")
    try:
        rec, brief = advise(p)
        advised.append({"payload": p, "rec": rec, "brief": brief})
        moved = sum(1 for t in rec["proposals"] if t["basis"] != "unchanged")
        print(f"-> {len(rec['proposals'])} proposals, {moved} change a limit")
    except Exception as e:
        rejected.append({"cde_id": p["cde_id"], "error": str(e)})
        print(f"-> REJECTED: {e}")

print(f"\n{len(advised)} elements advised, {len(rejected)} rejected")


In [ ]:
# Rejections FIRST and unconditionally -- the run where nothing validated is the run
# whose reason you most need (notebook 03 learned this the expensive way).
if rejected:
    print(f"{len(rejected)} element(s) rejected by validate() -- these wrote nothing:")
    for r in rejected:
        print(f"  {r['cde_id']}  {r['error']}")

if not advised:
    print("\nNothing was advised. Every rejection is listed above; each is a defect in "
          "the response or in the validator, not a reason to relax the gate.")
else:
    display(spark.createDataFrame([{
        "cde_id": a["payload"]["cde_id"], "rule_id": t["rule_id"],
        "current": next(m["current_threshold_pct"] for m in a["payload"]["members"]
                        if m["rule_id"] == t["rule_id"]),
        "proposed": float(t["proposed_threshold_pct"]), "basis": t["basis"],
        "tolerance": a["payload"]["tolerance_pct"], "reviewer": t["reviewer"],
    } for a in advised for t in a["rec"]["proposals"]]))


---
## 4 · Write

`INSERT`, never `MERGE`: a proposal is advice at a moment, and the reviewer's decision
references it by id. `proposal_id` is deterministic on the proposal run and the rule,
so re-running a failed write cannot double a proposal. `current_threshold_pct` and the
statistics come from the brief, never from the response.


In [ ]:
rows = []
for a in advised:
    p, rec = a["payload"], a["rec"]
    by_rule = {str(t["rule_id"]): t for t in rec["proposals"]}
    for m in p["members"]:
        t = by_rule[m["rule_id"]]
        h = m["history"] or {}
        rows.append({
            "proposal_id": det_uuid("threshold", PROPOSAL_RUN_ID, m["rule_id"]),
            "proposal_run_id": PROPOSAL_RUN_ID,
            "proposed_ts": PROPOSED_TS,
            "rule_id": m["rule_id"],
            "rule_version": m["rule_version"],
            "cde_id": p["cde_id"],
            "target_table": m["target_table"],
            "target_column": m["target_column"],
            "current_threshold_pct": float(m["current_threshold_pct"]),
            "proposed_threshold_pct": float(t["proposed_threshold_pct"]),
            "tolerance_pct": p["tolerance_pct"],
            "basis": t["basis"],
            "rationale": str(t["rationale"]),
            "reviewer": str(t["reviewer"]),
            "runs_observed": h.get("runs_observed"),
            "pct_min": h.get("pct_min"),
            "pct_median": h.get("pct_median"),
            "pct_p90": h.get("pct_p90"),
            "pct_max": h.get("pct_max"),
            "runs_breaching": h.get("runs_breaching"),
            "latest_violation_pct": m["latest_violation_pct"],
            "model_endpoint": MODEL_NAME,
            "model_input_payload": json.dumps({
                "reasoning": rec["reasoning"],
                "system_prompt_sha": hashlib.sha256(SYSTEM_PROMPT.encode()).hexdigest()[:16],
                "model": MODEL_NAME, "temperature": TEMPERATURE, "brief": a["brief"],
            }),
            "job_run_id": JOB_RUN_ID,
        })

# Schema from the table, not inferred -- a nullable column that is None in every row
# fails inference after every model call has been paid for.
candidates = spark.createDataFrame(rows, schema=spark.table(PROPOSAL).schema)
candidates.createOrReplaceTempView("candidate_proposals")
print(f"{len(rows)} proposals to write")

spark.sql(f"""
INSERT INTO {PROPOSAL}
SELECT c.* FROM candidate_proposals c
WHERE NOT EXISTS (SELECT 1 FROM {PROPOSAL} e WHERE e.proposal_id = c.proposal_id)
""")
print(spark.sql(f"SELECT COUNT(*) AS n FROM {PROPOSAL} WHERE proposal_run_id = '{PROPOSAL_RUN_ID}'")
      .collect()[0]["n"], "proposals on this run")


In [ ]:
# Verification: what the app's Thresholds page will show. Every proposal from this run
# is the latest for its rule, so each appears once, as open or no_change.
display(spark.sql(f"""
    SELECT rule_id, cde_id, current_threshold_pct, proposed_threshold_pct, tolerance_pct,
           basis, review_state, reviewer
    FROM   {PROPOSAL_CUR}
    WHERE  proposal_run_id = '{PROPOSAL_RUN_ID}'
    ORDER  BY review_state, rule_id
"""))
